# Amazon Reviews: NER and Sentiment Analysis

This notebook performs Named Entity Recognition (NER) to extract product names and brands, and analyzes sentiment using a rule-based approach.

## Step 1: Install Required Libraries

In [ ]:
!pip install -q spacy pandas numpy matplotlib seaborn textblob nltk

In [ ]:
# Download spaCy English model for NER
!python -m spacy download en_core_web_sm

In [ ]:
# Download NLTK data for sentiment analysis
import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

## Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# Load spaCy model for NER
nlp = spacy.load('en_core_web_sm')

# Initialize VADER sentiment analyzer
sia = SentimentIntensityAnalyzer()

print("Libraries loaded successfully!")

## Step 3: Load Sample Amazon Reviews Data

Note: Since the Kaggle dataset requires download, we'll create sample reviews for demonstration. In practice, you would load the actual dataset using:
```python
# For the actual Kaggle dataset:
# df = pd.read_csv('train.ft.txt', header=None, names=['label', 'review'])
```

In [ ]:
# Create sample Amazon reviews for demonstration
sample_reviews = [
    "I absolutely love my new iPhone 14 Pro! Apple really outdid themselves with the camera quality. Best purchase ever!",
    "The Samsung Galaxy S23 is terrible. Battery life is awful and it keeps freezing. Very disappointed with Samsung.",
    "This Sony WH-1000XM5 headphones are amazing! The noise cancellation is perfect for my daily commute. Highly recommend Sony products.",
    "Bought the Amazon Echo Dot and it's okay. Alexa works fine but the sound quality could be better. Not bad for the price.",
    "The Dell XPS 15 laptop is a beast! Intel Core i7 processor handles everything smoothly. Dell makes quality machines.",
    "Terrible experience with the HP Pavilion. It broke after 2 months. HP customer service was unhelpful. Don't buy!",
    "The Nike Air Max shoes are incredibly comfortable. Nike always delivers on quality. Worth every penny!",
    "This Kindle Paperwhite from Amazon is perfect for reading. The screen is easy on the eyes and battery lasts forever.",
    "The Microsoft Surface Pro is overpriced and underwhelming. Microsoft needs to do better. Not worth the money.",
    "Love my new MacBook Pro! Apple's M2 chip is lightning fast. Best laptop I've ever owned.",
    "The Google Pixel 7 camera is outstanding! Google's computational photography is unmatched. Excellent phone!",
    "Disappointed with the LG OLED TV. Picture quality is good but it has burn-in issues. LG should fix this.",
    "The Bose QuietComfort 45 headphones are worth every dollar. Bose continues to lead in audio quality.",
    "This Lenovo ThinkPad is solid for work. Lenovo builds reliable business laptops. Very satisfied.",
    "The Fitbit Charge 5 is inaccurate and uncomfortable. Fitbit quality has gone downhill. Returning it.",
    "Absolutely fantastic! The Canon EOS R5 camera is a professional's dream. Canon knows cameras.",
    "The Asus ROG gaming laptop runs hot and loud. Asus needs better cooling solutions. Disappointed.",
    "This Anker power bank is reliable and charges fast. Anker makes great accessories at good prices.",
    "The JBL Flip 6 speaker has amazing sound for its size. JBL never disappoints with portable audio.",
    "Terrible build quality on this Razer keyboard. Keys feel cheap. Razer is overpriced for what you get."
]

# Create DataFrame
df = pd.DataFrame({'review': sample_reviews})
df['review_id'] = range(1, len(df) + 1)

print(f"Loaded {len(df)} sample reviews")
print("\nFirst 5 reviews:")
print(df.head())

## Step 4: Named Entity Recognition (NER) - Extract Products and Brands

In [ ]:
def extract_entities(text):
    """
    Extract named entities from text using spaCy.
    Focus on ORG (organizations/brands) and PRODUCT entities.
    """
    doc = nlp(text)
    
    entities = {
        'brands': [],
        'products': [],
        'all_entities': []
    }
    
    for ent in doc.ents:
        # Extract organizations (brands)
        if ent.label_ == 'ORG':
            entities['brands'].append(ent.text)
        # Extract products
        elif ent.label_ == 'PRODUCT':
            entities['products'].append(ent.text)
        
        # Store all entities with their labels
        entities['all_entities'].append((ent.text, ent.label_))
    
    return entities

# Apply NER to all reviews
print("Performing Named Entity Recognition...")
df['entities'] = df['review'].apply(extract_entities)
df['brands'] = df['entities'].apply(lambda x: x['brands'])
df['products'] = df['entities'].apply(lambda x: x['products'])

print("\nNER completed!")
print("\nSample extracted entities:")
for idx in range(min(5, len(df))):
    print(f"\nReview {idx+1}: {df.iloc[idx]['review'][:80]}...")
    print(f"  Brands: {df.iloc[idx]['brands']}")
    print(f"  Products: {df.iloc[idx]['products']}")

## Step 5: Analyze Most Common Brands and Products

In [ ]:
# Count all brands mentioned
all_brands = [brand for brands_list in df['brands'] for brand in brands_list]
brand_counts = Counter(all_brands)

# Count all products mentioned
all_products = [product for products_list in df['products'] for product in products_list]
product_counts = Counter(all_products)

print("Most Common Brands:")
print("=" * 50)
for brand, count in brand_counts.most_common(10):
    print(f"{brand}: {count} mentions")

print("\nMost Common Products:")
print("=" * 50)
for product, count in product_counts.most_common(10):
    print(f"{product}: {count} mentions")

print(f"\nTotal unique brands: {len(brand_counts)}")
print(f"Total unique products: {len(product_counts)}")

In [ ]:
# Visualize top brands
if len(brand_counts) > 0:
    top_brands = dict(brand_counts.most_common(10))
    
    plt.figure(figsize=(12, 6))
    plt.bar(top_brands.keys(), top_brands.values(), color='skyblue', edgecolor='black')
    plt.xlabel('Brand', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.title('Top 10 Most Mentioned Brands in Reviews', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No brands extracted. This may happen with the spaCy model's entity recognition.")

## Step 6: Rule-Based Sentiment Analysis

In [ ]:
def analyze_sentiment_vader(text):
    """
    Analyze sentiment using VADER (Valence Aware Dictionary and sEntiment Reasoner).
    VADER is a rule-based sentiment analyzer specifically tuned for social media text.
    """
    scores = sia.polarity_scores(text)
    
    # Compound score: normalized score between -1 (most negative) and +1 (most positive)
    compound = scores['compound']
    
    # Classify sentiment based on compound score
    if compound >= 0.05:
        sentiment = 'Positive'
    elif compound <= -0.05:
        sentiment = 'Negative'
    else:
        sentiment = 'Neutral'
    
    return {
        'sentiment': sentiment,
        'compound': compound,
        'positive': scores['pos'],
        'negative': scores['neg'],
        'neutral': scores['neu']
    }

def analyze_sentiment_textblob(text):
    """
    Analyze sentiment using TextBlob.
    TextBlob provides polarity (-1 to 1) and subjectivity (0 to 1) scores.
    """
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity
    
    if polarity > 0.1:
        sentiment = 'Positive'
    elif polarity < -0.1:
        sentiment = 'Negative'
    else:
        sentiment = 'Neutral'
    
    return {
        'sentiment': sentiment,
        'polarity': polarity,
        'subjectivity': subjectivity
    }

# Apply both sentiment analysis methods
print("Performing sentiment analysis...")
df['vader_sentiment'] = df['review'].apply(analyze_sentiment_vader)
df['textblob_sentiment'] = df['review'].apply(analyze_sentiment_textblob)

# Extract sentiment labels
df['vader_label'] = df['vader_sentiment'].apply(lambda x: x['sentiment'])
df['textblob_label'] = df['textblob_sentiment'].apply(lambda x: x['sentiment'])
df['vader_score'] = df['vader_sentiment'].apply(lambda x: x['compound'])
df['textblob_score'] = df['textblob_sentiment'].apply(lambda x: x['polarity'])

print("Sentiment analysis completed!")

## Step 7: Display Results - Extracted Entities and Sentiment

In [ ]:
# Create a comprehensive results dataframe
results_df = df[['review_id', 'review', 'brands', 'products', 'vader_label', 'vader_score', 'textblob_label', 'textblob_score']].copy()

print("EXTRACTED ENTITIES AND SENTIMENT ANALYSIS RESULTS")
print("=" * 100)

for idx, row in results_df.iterrows():
    print(f"\n{'='*100}")
    print(f"Review #{row['review_id']}")
    print(f"{'='*100}")
    print(f"Text: {row['review']}")
    print(f"\nExtracted Entities:")
    print(f"  - Brands: {', '.join(row['brands']) if row['brands'] else 'None detected'}")
    print(f"  - Products: {', '.join(row['products']) if row['products'] else 'None detected'}")
    print(f"\nSentiment Analysis:")
    print(f"  - VADER: {row['vader_label']} (score: {row['vader_score']:.3f})")
    print(f"  - TextBlob: {row['textblob_label']} (score: {row['textblob_score']:.3f})")

print(f"\n{'='*100}")

## Step 8: Sentiment Distribution Analysis

In [ ]:
# Count sentiment distribution
vader_counts = df['vader_label'].value_counts()
textblob_counts = df['textblob_label'].value_counts()

print("Sentiment Distribution:")
print("=" * 50)
print("\nVADER Sentiment:")
for sentiment, count in vader_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  {sentiment}: {count} ({percentage:.1f}%)")

print("\nTextBlob Sentiment:")
for sentiment, count in textblob_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  {sentiment}: {count} ({percentage:.1f}%)")

In [ ]:
# Visualize sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# VADER sentiment distribution
colors = {'Positive': 'green', 'Negative': 'red', 'Neutral': 'gray'}
vader_colors = [colors.get(x, 'blue') for x in vader_counts.index]
axes[0].bar(vader_counts.index, vader_counts.values, color=vader_colors, edgecolor='black', alpha=0.7)
axes[0].set_title('VADER Sentiment Distribution', fontsize=14)
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)

# TextBlob sentiment distribution
textblob_colors = [colors.get(x, 'blue') for x in textblob_counts.index]
axes[1].bar(textblob_counts.index, textblob_counts.values, color=textblob_colors, edgecolor='black', alpha=0.7)
axes[1].set_title('TextBlob Sentiment Distribution', fontsize=14)
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Count')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Step 9: Sentiment Score Distribution

In [ ]:
# Plot sentiment score distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# VADER compound scores
axes[0].hist(df['vader_score'], bins=20, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Neutral (0)')
axes[0].set_title('VADER Compound Score Distribution', fontsize=14)
axes[0].set_xlabel('Compound Score')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# TextBlob polarity scores
axes[1].hist(df['textblob_score'], bins=20, color='lightcoral', edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Neutral (0)')
axes[1].set_title('TextBlob Polarity Score Distribution', fontsize=14)
axes[1].set_xlabel('Polarity Score')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10: Brand-Specific Sentiment Analysis

In [ ]:
# Analyze sentiment for each brand
brand_sentiment = []

for idx, row in df.iterrows():
    for brand in row['brands']:
        brand_sentiment.append({
            'brand': brand,
            'sentiment': row['vader_label'],
            'score': row['vader_score']
        })

if brand_sentiment:
    brand_sentiment_df = pd.DataFrame(brand_sentiment)
    
    print("Brand-Specific Sentiment Analysis:")
    print("=" * 70)
    
    # Calculate average sentiment score per brand
    brand_avg_sentiment = brand_sentiment_df.groupby('brand').agg({
        'score': ['mean', 'count']
    }).round(3)
    brand_avg_sentiment.columns = ['avg_sentiment_score', 'mention_count']
    brand_avg_sentiment = brand_avg_sentiment.sort_values('avg_sentiment_score', ascending=False)
    
    print("\nAverage Sentiment Score by Brand:")
    print(brand_avg_sentiment)
    
    # Visualize brand sentiment
    if len(brand_avg_sentiment) > 0:
        plt.figure(figsize=(12, 6))
        colors_map = brand_avg_sentiment['avg_sentiment_score'].apply(
            lambda x: 'green' if x > 0.05 else ('red' if x < -0.05 else 'gray')
        )
        plt.barh(brand_avg_sentiment.index, brand_avg_sentiment['avg_sentiment_score'], 
                color=colors_map, edgecolor='black', alpha=0.7)
        plt.xlabel('Average Sentiment Score', fontsize=12)
        plt.ylabel('Brand', fontsize=12)
        plt.title('Average Sentiment Score by Brand', fontsize=14)
        plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("No brands detected for sentiment analysis.")

## Step 11: Summary Statistics

In [ ]:
# Generate comprehensive summary
print("SUMMARY STATISTICS")
print("=" * 70)

print(f"\nTotal Reviews Analyzed: {len(df)}")
print(f"\nEntity Extraction:")
print(f"  - Total Brands Mentioned: {len(all_brands)}")
print(f"  - Unique Brands: {len(brand_counts)}")
print(f"  - Total Products Mentioned: {len(all_products)}")
print(f"  - Unique Products: {len(product_counts)}")

print(f"\nSentiment Analysis (VADER):")
for sentiment in ['Positive', 'Negative', 'Neutral']:
    count = vader_counts.get(sentiment, 0)
    percentage = (count / len(df)) * 100
    print(f"  - {sentiment}: {count} ({percentage:.1f}%)")

print(f"\nSentiment Score Statistics (VADER):")
print(f"  - Mean: {df['vader_score'].mean():.3f}")
print(f"  - Median: {df['vader_score'].median():.3f}")
print(f"  - Std Dev: {df['vader_score'].std():.3f}")
print(f"  - Min: {df['vader_score'].min():.3f}")
print(f"  - Max: {df['vader_score'].max():.3f}")

print(f"\nMost Positive Review (VADER):")
most_positive_idx = df['vader_score'].idxmax()
print(f"  Score: {df.loc[most_positive_idx, 'vader_score']:.3f}")
print(f"  Review: {df.loc[most_positive_idx, 'review']}")

print(f"\nMost Negative Review (VADER):")
most_negative_idx = df['vader_score'].idxmin()
print(f"  Score: {df.loc[most_negative_idx, 'vader_score']:.3f}")
print(f"  Review: {df.loc[most_negative_idx, 'review']}")

## Step 12: Export Results

In [ ]:
# Prepare export dataframe
export_df = df[['review_id', 'review', 'brands', 'products', 'vader_label', 'vader_score', 'textblob_label', 'textblob_score']].copy()

# Convert lists to strings for CSV export
export_df['brands'] = export_df['brands'].apply(lambda x: ', '.join(x) if x else '')
export_df['products'] = export_df['products'].apply(lambda x: ', '.join(x) if x else '')

# Save to CSV
export_df.to_csv('amazon_reviews_analysis.csv', index=False)
print("Results exported to 'amazon_reviews_analysis.csv'")

# Display sample of export
print("\nSample of exported data:")
print(export_df.head())

## Summary

### Methodology:

**Named Entity Recognition (NER):**
- Used spaCy's pre-trained English model (`en_core_web_sm`)
- Extracted ORG entities (brands/organizations)
- Extracted PRODUCT entities (product names)
- Identified and counted unique brands and products

**Sentiment Analysis (Rule-Based):**
- **VADER (Valence Aware Dictionary and sEntiment Reasoner)**:
  - Rule-based sentiment analyzer
  - Specifically tuned for social media and review text
  - Provides compound score (-1 to +1)
  - Classification: Positive (>0.05), Negative (<-0.05), Neutral (between)

- **TextBlob**:
  - Pattern-based sentiment analysis
  - Provides polarity (-1 to +1) and subjectivity (0 to 1)
  - Good for general text sentiment

### Key Features:
1. Automatic entity extraction from reviews
2. Dual sentiment analysis for comparison
3. Brand-specific sentiment tracking
4. Comprehensive visualizations
5. Detailed statistics and summaries
6. Exportable results

### Use Cases:
- Product reputation monitoring
- Brand sentiment tracking
- Customer feedback analysis
- Competitive analysis
- Market research

### Note:
For the actual Kaggle Amazon Reviews dataset, load it using:
```python
df = pd.read_csv('train.ft.txt', header=None, names=['label', 'review'])
```
The dataset format is: `__label__X review_text` where X is 1 (negative) or 2 (positive).